# Philippine government sentiment study - 2026

This notebook runs the supplied **Bluesky and Mastodon curated, labeled CSV snapshot** offline. Download the whole uncompressed repository, install the project following `README.md`, select that Python environment, and choose **Run All**. No social-media account or API token is needed. Keep `src/`, `pyproject.toml`, and `deliverables/labeled_2026/` together.

The supplied reference labels were created through AI-assisted review of the posts and available public context, independently of the fixed rule classifier. **Evaluation measures agreement with AI-assisted reference labels, not human-validated accuracy.** The annotation decisions, reasons, context, independent AI review, and adjudication are supplied for audit. The recorded AI-to-AI agreement describes annotation consistency; it does not establish human-validated correctness. The `gold_label` column name is retained for schema compatibility; it does not imply human gold-standard validation.

**Remaining group work:** name and interpret the topics, inspect actual proximity and evaluation results, document the collection/annotation methods and limitations, and complete the report and peer evaluation. Labeling and computational evaluation run from the supplied files by default. Optional independent human validation would strengthen subsequent claims.

Use `deliverables/labeled_2026/README.md` and `docs/methodology.md` for the study design.

**Interpretation boundary:** the reference labels capture target-related polarity in the content, including favorable or adverse governance claims in reporting. They do not establish the author's personal approval. Corpus sentiment is not a population approval estimate or an objective measurement of government performance.

## 1. Setup and reproducibility

The notebook locates the repository from its root or any subdirectory, verifies the complete snapshot SHA-256 manifest, and writes working outputs beneath `data/labeled-notebook-run/`. Keep the supplied snapshot unchanged. The earlier candidate collection and validation snapshots remain historical records.

This workflow copies the already curated corpus and supplied labels, then performs analysis and evaluation. It makes no live collection requests. It deliberately does not prepare the full raw collection again, which would reintroduce excluded candidates.

Repeat runs reuse complete outputs only when their recorded input and output hashes match. Changed inputs, configuration, implementation, or partial outputs produce an actionable error. Set `FORCE_REBUILD = True` to regenerate derived outputs intentionally. Existing working annotations are preserved. For separately completed human annotations, set `LABEL_SOURCE = "human"` and retain their documented provenance.

In [1]:
import csv
import hashlib
import html
import shutil
import subprocess
import sys
from collections import Counter
from dataclasses import replace
from importlib.metadata import version
from pathlib import Path

from IPython.display import HTML, display

from ph_sentiment.csvio import read_records, write_csv


def find_repository() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the repository or its notebooks directory.")


REPO = find_repository()
SNAPSHOT = REPO / "deliverables" / "labeled_2026"
CONFIG = SNAPSHOT / "config.toml"
PREPARED_SOURCE = SNAPSHOT / "processed" / "posts.csv"
ANNOTATION_SOURCE = SNAPSHOT / "labels" / "annotation.csv"
DATA_DIR = REPO / "data" / "labeled-notebook-run"
PROCESSED = DATA_DIR / "processed" / "posts.csv"
ANALYSIS_DIR = DATA_DIR / "analysis"
ANNOTATIONS = DATA_DIR / "labels" / "annotation.csv"
EVALUATION_DIR = DATA_DIR / "evaluation"
FORCE_REBUILD = False
LABEL_SOURCE = "ai_assisted"

if LABEL_SOURCE not in {"ai_assisted", "human"}:
    raise ValueError("LABEL_SOURCE must be ai_assisted or human.")
missing_inputs = [
    path
    for path in (CONFIG, PREPARED_SOURCE, ANNOTATION_SOURCE, SNAPSHOT / "file_manifest.csv")
    if not path.is_file()
]
if missing_inputs:
    missing = ", ".join(str(path.relative_to(REPO)) for path in missing_inputs)
    raise FileNotFoundError(f"Handoff files are missing: {missing}. Download the full repository.")
if DATA_DIR.resolve().is_relative_to(SNAPSHOT.resolve()):
    raise ValueError("Keep working outputs outside the supplied snapshot directory.")


def file_digest(path: Path) -> str:
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def preview_csv(
    path: Path,
    limit: int = 6,
    max_columns: int = 8,
    where: dict[str, str] | None = None,
) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Required output is missing: {path.relative_to(REPO)}")
    rows = read_csv(path)
    total_rows = len(rows)
    if where:
        if rows and not set(where).issubset(rows[0]):
            raise ValueError("Preview filter references a missing CSV column.")
        rows = [row for row in rows if all(row.get(name) == value for name, value in where.items())]
        print(f"{path.relative_to(REPO)}: {len(rows):,} matching rows of {total_rows:,}")
    else:
        print(f"{path.relative_to(REPO)}: {total_rows:,} rows")
    if not rows:
        print("No rows to preview.")
        return
    columns = list(rows[0])[:max_columns]
    headings = "".join(f'<th scope="col">{html.escape(name)}</th>' for name in columns)
    body = []
    for row in rows[:limit]:
        values = []
        for name in columns:
            value = str(row.get(name, ""))
            if len(value) > 180:
                value = value[:180] + "..."
            values.append(f"<td>{html.escape(value)}</td>")
        body.append("<tr>" + "".join(values) + "</tr>")
    caption = html.escape(
        f"{path.name}: first {min(limit, len(rows))} matching rows; first {len(columns)} columns"
    )
    display(
        HTML(
            f"<table><caption>{caption}</caption><thead><tr>{headings}</tr></thead>"
            f"<tbody>{''.join(body)}</tbody></table>"
        )
    )


def run_cli(*arguments: str) -> None:
    if not arguments or arguments[0] not in {"analyze", "evaluate"}:
        raise ValueError("The labeled handoff permits offline analysis/evaluation commands only.")
    result = subprocess.run(
        [sys.executable, "-m", "ph_sentiment", *map(str, arguments)],
        cwd=REPO,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode:
        detail = result.stderr.strip() or "No diagnostic was returned."
        raise RuntimeError(f"Pipeline command failed ({result.returncode}): {detail}")
    if result.stderr.strip():
        print(result.stderr.strip())


def stage_fingerprint(stage: str, inputs: tuple[Path, ...], extra: str = "") -> str:
    # Include implementation hashes: a code change must not silently reuse old results.
    modules = (
        "schema.py",
        "csvio.py",
        "preprocessing.py",
        "sentiment.py",
        "cli.py",
        "config.py",
        f"{stage}.py",
    )
    parts = ["labeled-notebook-v2", stage, extra, sys.version]
    parts.extend(f"{name}={version(name)}" for name in ("numpy", "scipy", "scikit-learn"))
    parts.extend(file_digest(path) for path in inputs)
    parts.extend(file_digest(REPO / "src" / "ph_sentiment" / name) for name in modules)
    return hashlib.sha256("\n".join(parts).encode("utf-8")).hexdigest()


def cached_stage(stage: str, outputs: tuple[Path, ...], fingerprint: str) -> bool:
    state = DATA_DIR / "run_state" / f"{stage}.csv"
    if FORCE_REBUILD:
        return False
    if not state.exists() and not any(path.exists() for path in outputs):
        return False
    if state.is_file() and all(path.is_file() for path in outputs):
        rows = read_csv(state)
        expected = {path.relative_to(DATA_DIR).as_posix(): file_digest(path) for path in outputs}
        recorded = {row["path"]: row["sha256"] for row in rows}
        if (
            len(rows) == len(expected)
            and recorded == expected
            and all(row["input_fingerprint"] == fingerprint for row in rows)
        ):
            print(f"Reusing verified {stage} outputs for these inputs.")
            return True
    raise RuntimeError(
        f"{stage.capitalize()} inputs or outputs changed, or the previous run is incomplete. "
        "Set FORCE_REBUILD = True to regenerate derived CSVs; working labels are preserved."
    )


def remember_stage(stage: str, outputs: tuple[Path, ...], fingerprint: str) -> None:
    write_csv(
        DATA_DIR / "run_state" / f"{stage}.csv",
        [
            {
                "path": path.relative_to(DATA_DIR).as_posix(),
                "sha256": file_digest(path),
                "input_fingerprint": fingerprint,
            }
            for path in outputs
        ],
        ("path", "sha256", "input_fingerprint"),
    )


def verify_annotation_records(
    path: Path, expected: list, *, require_supplied: bool = False
) -> list:
    records = read_records(path)
    for record in records:
        record.validate()
        if not record.gold_label:
            raise ValueError(f"Incomplete reference label in {path.relative_to(REPO)}.")
    expected_map = {(record.platform, record.record_id): record for record in expected}
    actual_map = {(record.platform, record.record_id): record for record in records}
    if len(actual_map) != len(records) or actual_map.keys() != expected_map.keys():
        raise ValueError("Annotation IDs/platforms must match the curated corpus exactly.")
    for key, record in actual_map.items():
        if replace(record, gold_label="") != replace(expected_map[key], gold_label=""):
            raise ValueError(
                "Annotation text/provenance changed. Preserve labels and reconcile the corpus."
            )
        if require_supplied and record.gold_label != expected_map[key].gold_label:
            raise ValueError("Supplied annotation labels do not match the curated corpus.")
    return records


# The manifest must cover every snapshot file except itself, with no duplicate/unsafe paths.
manifest = read_csv(SNAPSHOT / "file_manifest.csv")
paths = [entry["path"] for entry in manifest]
actual_paths = {
    path.relative_to(SNAPSHOT).as_posix()
    for path in SNAPSHOT.rglob("*")
    if path.is_file() and path.name != "file_manifest.csv"
}
if len(set(paths)) != len(paths) or set(paths) != actual_paths:
    raise ValueError("The snapshot manifest is incomplete or contains duplicate entries.")
for entry in manifest:
    supplied_file = (SNAPSHOT / entry["path"]).resolve()
    if not supplied_file.is_relative_to(SNAPSHOT.resolve()) or not supplied_file.is_file():
        raise ValueError(f"Invalid or missing snapshot file: {entry['path']}")
    if file_digest(supplied_file) != entry["sha256"] or supplied_file.stat().st_size != int(
        entry["bytes"]
    ):
        raise ValueError(
            f"Snapshot checksum/size mismatch: {entry['path']}. Restore the supplied file."
        )
    if supplied_file.suffix == ".csv" and len(read_csv(supplied_file)) != int(entry["rows"]):
        raise ValueError(f"Snapshot row-count mismatch: {entry['path']}")
print("Complete supplied snapshot checksums verified.")
print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Working directory: {DATA_DIR.relative_to(REPO)}")
print("Offline handoff: curated reference labels supplied; no live collection requests.")
print("AI-assisted reference agreement is not human-validated accuracy.")

Complete supplied snapshot checksums verified.
Python: 3.12.10 (C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\.venv\Scripts\python.exe)
Working directory: data\labeled-notebook-run
Offline handoff: curated reference labels supplied; no live collection requests.
AI-assisted reference agreement is not human-validated accuracy.


## 2. Inspect collection, curation, and annotation provenance

The snapshot preserves the returned public records, available embed context, inclusion/exclusion decisions, and concise annotation reasons. The study window begins **2026-01-01** and ends before **2027-01-01**; collection covers only posts available at collection time. Read the observed source/date counts rather than assuming complete-year or balanced coverage.

The curated CSV is the analysis input. Original raw files are retained for audit; excluded or unresolved candidates are not silently relabeled neutral or reintroduced. Source and curation summaries below report the actual delivered counts. No X or Threads observations are included.

Two initial AI assistants annotated disjoint batches without seeing rule outputs. A third AI assistant independently reviewed a simple random sample of 60 candidates (seed `20261012`), followed by root AI adjudication. `labels/annotation_quality.csv` reports the measured review agreement; `independent_review.csv` and `adjudication.csv` retain the decisions. This is consistency among AI reviewers, not human validation. See `docs/annotation_protocol.md`.

`annotation_summary.csv` describes the supplied annotations. `labels/reference_sentiment_summary.csv` groups their target-related polarity by topic, source, and content purpose. This reference summary differs from `analysis/sentiment_summary.csv`, which counts the fixed rule classifier's predictions. Keep their label sources explicit when reporting either result.

In [2]:
for filename in ("source_summary.csv", "curation_summary.csv", "annotation_summary.csv"):
    preview_csv(SNAPSHOT / filename)
preview_csv(SNAPSHOT / "labels" / "annotation_quality.csv", max_columns=12)
preview_csv(SNAPSHOT / "labels" / "annotation_decisions.csv", max_columns=12)
preview_csv(SNAPSHOT / "labels" / "independent_review.csv", max_columns=12)
preview_csv(SNAPSHOT / "labels" / "adjudication.csv", max_columns=12)
preview_csv(SNAPSHOT / "labels" / "post_context.csv", max_columns=10)
for source in sorted((SNAPSHOT / "raw").glob("*.csv")):
    print(f"Supplied raw/audit file: {source.relative_to(REPO)} ({len(read_csv(source)):,} rows)")

deliverables\labeled_2026\source_summary.csv: 2 rows


platform,raw_records,prepared_candidates,curated_records,negative,neutral,positive,first_post_date_manila
bluesky,289,261,182,82,86,14,2026-08-06
mastodon,75,74,49,22,24,3,2026-02-04


deliverables\labeled_2026\curation_summary.csv: 3 rows


relevance,records,label_source,notes
include,231,ai_assisted,Included posts have resolved reference labels; excluded/unresolved posts have no sentiment label.
exclude,96,ai_assisted,Included posts have resolved reference labels; excluded/unresolved posts have no sentiment label.
unresolved,8,ai_assisted,Included posts have resolved reference labels; excluded/unresolved posts have no sentiment label.


deliverables\labeled_2026\annotation_summary.csv: 3 rows


...

deliverables\labeled_2026\labels\adjudication.csv: 7 rows


index,record_id,platform,primary_relevance,primary_label,review_relevance,review_label,final_relevance,final_label,reason,adjudicator,annotation_date
2,bluesky:at://did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/app.bsky.feed.post/3mxnufw5l522l,bluesky,include,positive,include,neutral,include,neutral,Reports a completed visit and future investment plans; the preview does not establish delivered benefits or explicit independent endorsement of government performance.,root_ai_adjudication,2026-10-12
42,bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mx5sqprpbs2m,bluesky,exclude,,include,neutral,exclude,,"A president's personal enjoyment of music does not discuss governance, public policy or government performance.",root_ai_adjudication,2026-10-12
45,bluesky:at://did:plc:g25lz3c7n37c4vofgm4exohj/app.bsky.feed.post/3mx5hutalhk2u,bluesky,include,negative,unresolved,,unresolved,,The analogy names Marcos without identifying Sr. or Jr.; available text cannot establish a current 2026 national-government target.,root_ai_adjudication,2026-10-12
87,bluesky:at://did:plc:jyjjqbfoetzn5kmh6sjiauc4/app.bsky.feed.post/3mxfvqddzc22y,bluesky,include,neutral,include,positive,include,positive,"Credits the Philippine government's peace agreement with enabling actual first elections after prolonged deadly conflict, a beneficial national-policy outcome.",root_ai_adjudication,2026-10-12
107,bluesky:at://did:plc:w3tm4xu2rkt7jng2tfhj2py7/app.bsky.feed.post/3mwz3pnhdly2d,bluesky,include,positive,not_sampled,,exclude,,A promotional government-bond product advertisement without discussion of governance or government performance; outside the political research target.,root_ai_adjudication,2026-10-12
150,bluesky:at://did:plc:25bhka7cek7uxvshdcdtbish/app.bsky.feed.post/3mwatszevds2x,bluesky,include,negative,unresolved,,unresolved,,The fragment could concern government disbanding a party or advocate disbanding government; target evaluation cannot be resolved.,root_ai_adjudication,2026-10-12


deliverables\labeled_2026\labels\post_context.csv: 335 rows


record_id,platform,context_status,snapshot_text_matches,current_text,link_title,link_description,image_alt_text,quoted_text,collected_at
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,available,true,"#The US and Japan are violating the Philippines' sovereignty In recent years, the Marcos administration has raised the banner of ""Armed Forces Modernization",,,,,2026-10-12T08:43:53.392672Z
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx62vedk2a,bluesky,available,true,"#Political Battles Are Tearing The Philippines Apart The Illusion of Arms Procurement: How the Marcos Administration Drains the National Treasury in the Name of ""Military Moderniz...",,,,,2026-10-12T08:43:53.392672Z
bluesky:at://did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/app.bsky.feed.post/3mxnufw5l522l,bluesky,available,true,President Marcos returns from Singapore visit focused on business expansion and energy investments 🤖 IA: It's not clickbait ✅ 👥 Users: It's not clickbait ✅ #marcos #singapore #in...,President Marcos returns from Singapore visit focused on business expansion and energy investments,"President Ferdinand 'Bongbong' Marcos Jr. has returned to the Philippines after completing a working visit to Singapore from October 7 to 11, 2026. During his trip, he engaged with...",,,2026-10-12T08:43:53.392672Z
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxnqzmxal322,bluesky,available,true,Marcos Orders Scrutiny of Alleged P319.33 Million China Remittances Linked to Duterte's Husband #Philippines #Politics,Marcos Orders Scrutiny of Alleged P319.33 Million China Remittances Linked to Duterte's Husband,P319.33 Million in Transfers Draw Security Scrutiny Alleged remittances totaling P319.33 million from mainland China and Hong Kong to a food company linked to Vice President Sara D...,,,2026-10-12T08:43:53.392672Z
bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxnek4papo2w,bluesky,available,true,"Japanese Ambassador Katsuro Kitagawa presented his Letters of Credence to President Ferdinand Marcos Jr. on Oct. 1, formally beginning his tenure as Japan’s ambassador to the Phili...",,,,,2026-10-12T08:43:53.392672Z
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxn2o72ay222,bluesky,available,true,Marcos Rejects Chinese Permission Demands After Nearly 10 Hours to Evacuate Ayungin Sailor #China #Philippines #Politics,Marcos Rejects Chinese Permission Demands After Nearly 10 Hours to Evacuate Ayungin Sailor,"A medical emergency becomes a maritime confrontation A Philippine Navy sailor needing urgent medical care was transferred from the BRP Sierra Madre at Ayungin Shoal on October 8, 2...",,,2026-10-12T08:43:53.392672Z


Supplied raw/audit file: deliverables\labeled_2026\raw\bluesky.csv (289 rows)
Supplied raw/audit file: deliverables\labeled_2026\raw\bluesky_collection_manifest.csv (1 rows)
Supplied raw/audit file: deliverables\labeled_2026\raw\bluesky_query_diagnostics.csv (0 rows)
Supplied raw/audit file: deliverables\labeled_2026\raw\mastodon.csv (75 rows)
Supplied raw/audit file: deliverables\labeled_2026\raw\mastodon_collection_manifest.csv (1 rows)
Supplied raw/audit file: deliverables\labeled_2026\raw\mastodon_query_diagnostics.csv (0 rows)


## 3. Use the curated corpus

This stage copies `deliverables/labeled_2026/processed/posts.csv` into the working directory, retaining its supplied reference labels. Cleaning and relevance curation have already been performed; the full raw candidate collection is preserved separately. The decision CSV records why candidates were included, excluded, or unresolved.

The working prepared file must match the supplied copy. A changed or stale file causes an error unless rebuilding is explicitly enabled. Working annotations remain protected. For a separately revised corpus, create a new versioned snapshot instead of quietly mixing incompatible records and labels.

In [3]:
if (
    PROCESSED.exists()
    and file_digest(PROCESSED) != file_digest(PREPARED_SOURCE)
    and not FORCE_REBUILD
):
    raise RuntimeError(
        "Working corpus differs from the supplied curated CSV. Set FORCE_REBUILD = True to restore it."
    )
if not PROCESSED.exists() or FORCE_REBUILD:
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PREPARED_SOURCE, PROCESSED)
prepared_records = read_records(PROCESSED)
for record in prepared_records:
    record.validate()
verify_annotation_records(ANNOTATION_SOURCE, prepared_records, require_supplied=True)
print("Curated working corpus matches the supplied snapshot.")
preview_csv(PROCESSED)

Curated working corpus matches the supplied snapshot.
data\labeled-notebook-run\processed\posts.csv: 231 rows


record_id,platform,created_at,text,source_url,query,record_type,collected_at
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,2026-10-12T07:02:57.396000+00:00,"the us and japan are violating the philippines' sovereignty in recent years, the marcos administration has raised the banner of ""armed forces modernization",https://bsky.app/profile/did:plc:bmnkh5p5evoqb7iwrx362p4t/post/3mxnx6uq7d22a,Philippines Marcos,post,2026-10-12T07:37:19.329647Z
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx62vedk2a,bluesky,2026-10-12T07:02:30.302000+00:00,"political battles are tearing the philippines apart the illusion of arms procurement: how the marcos administration drains the national treasury in the name of ""military modernizat...",https://bsky.app/profile/did:plc:bmnkh5p5evoqb7iwrx362p4t/post/3mxnx62vedk2a,Philippines Marcos,post,2026-10-12T07:37:19.329647Z
bluesky:at://did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/app.bsky.feed.post/3mxnufw5l522l,bluesky,2026-10-12T06:13:12.126989+00:00,president marcos returns from singapore visit focused on business expansion and energy investments 🤖 ia: it's not clickbait ✅ 👥 users: it's not clickbait ✅ marcos singapore investm...,https://bsky.app/profile/did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/post/3mxnufw5l522l,Philippines Marcos,post,2026-10-12T07:37:19.329647Z
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxnqzmxal322,bluesky,2026-10-12T05:12:39+00:00,marcos orders scrutiny of alleged p319.33 million china remittances linked to duterte's husband philippines politics,https://bsky.app/profile/did:plc:jhgdghatjzlq2nzwawreyfll/post/3mxnqzmxal322,Philippines Marcos,post,2026-10-12T07:37:19.329647Z
bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxnek4papo2w,bluesky,2026-10-12T01:29:12.807000+00:00,"japanese ambassador katsuro kitagawa presented his letters of credence to president ferdinand marcos jr. on oct. 1, formally beginning his tenure as japan's ambassador to the phili...",https://bsky.app/profile/did:plc:qvidzz7dm76lanyikijwp4c4/post/3mxnek4papo2w,Philippines Marcos,post,2026-10-12T07:37:19.329647Z
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxn2o72ay222,bluesky,2026-10-11T22:32:33+00:00,marcos rejects chinese permission demands after nearly 10 hours to evacuate ayungin sailor china philippines politics,https://bsky.app/profile/did:plc:jhgdghatjzlq2nzwawreyfll/post/3mxn2o72ay222,Philippines Marcos,post,2026-10-12T07:37:19.329647Z


In [4]:
prepared_rows = read_csv(PROCESSED)
platform_counts = Counter(record.platform for record in prepared_records)
label_counts = Counter(record.gold_label for record in prepared_records)
print(f"Curated labeled records: {len(prepared_records):,}")
print(f"Actual source counts: {dict(sorted(platform_counts.items()))}")
print(f"Supplied AI-assisted reference counts: {dict(sorted(label_counts.items()))}")
if len(prepared_records) < 200 or len(platform_counts) < 2:
    raise ValueError(
        "The activity requires at least 200 curated unique records from different sources."
    )
print("Count/source checks passed. Review the documented coverage and annotation limitations.")

Curated labeled records: 231
Actual source counts: {'bluesky': 182, 'mastodon': 49}
Supplied AI-assisted reference counts: {'negative': 104, 'neutral': 110, 'positive': 17}
Count/source checks passed. Review the documented coverage and annotation limitations.


### Document the supplied quality review

Use the source, curation, decision, and context CSVs to complete these research notes:

- Collection time, search vocabulary, endpoint coverage, and observed date interval:
- Curated counts by platform and documented exclusion/unresolved reasons:
- AI-assisted annotation protocol, available context, and remaining uncertainty:
- Source/language/month imbalance and suspected near-duplicate content:

Keep the historical `public_2026` and `validation_2026` results distinct from the curated labeled run.

## 4. Load the supplied reference labels

`deliverables/labeled_2026/labels/annotation.csv` supplies complete AI-assisted reference labels. This cell copies them into `data/labeled-notebook-run/labels/annotation.csv` when absent and preserves an existing working file. IDs, text, provenance, and supplied labels are checked against the curated corpus before the first copy.

The annotation target is **polarity of content about the Philippine national administration or its governance during 2026**:

- **positive:** favorable assessment, approval, a claimed governance success, or a beneficial government action attributed to the target.
- **negative:** criticism, disapproval, alleged misconduct, governance failure, or an adverse government action attributed to the target.
- **neutral:** target-related information without a sufficiently clear favorable/adverse assessment, including balanced or mixed content with no dominant polarity.

News reporting can carry target-related favorable/adverse claims without showing the posting author's personal approval. Unresolved target relevance is excluded, rather than filled with a neutral label. Audit reasons and public context are retained separately. These labels were not produced by copying rule predictions. The independent random-sample AI review and root adjudication are documented in the supplied CSVs and `docs/annotation_protocol.md`; measured AI agreement is a consistency diagnostic, not human accuracy.

Optional human validation can replace the working reference labels while preserving IDs/text; set `LABEL_SOURCE = "human"` only after documenting that review. The default handoff requires no additional manual labeling.

In [5]:
supplied_records = verify_annotation_records(
    ANNOTATION_SOURCE, prepared_records, require_supplied=True
)
if not ANNOTATIONS.exists():
    ANNOTATIONS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ANNOTATION_SOURCE, ANNOTATIONS)
    print(f"Supplied reference labels copied to: {ANNOTATIONS.relative_to(REPO)}")
else:
    print(f"Preserving existing working labels: {ANNOTATIONS.relative_to(REPO)}")
annotation_records = verify_annotation_records(ANNOTATIONS, prepared_records)
if LABEL_SOURCE == "ai_assisted" and {
    (record.platform, record.record_id): record.gold_label for record in annotation_records
} != {(record.platform, record.record_id): record.gold_label for record in supplied_records}:
    raise ValueError(
        "Working labels differ from the supplied AI-assisted references. They are preserved. "
        "Document a human review and use LABEL_SOURCE = 'human', or restore the supplied working copy."
    )
print(f"Reference-label source: {LABEL_SOURCE}")
print(
    f"Working reference counts: {dict(sorted(Counter(record.gold_label for record in annotation_records).items()))}"
)
print("Labels are complete; analysis and evaluation can run without group annotation.")

Supplied reference labels copied to: data\labeled-notebook-run\labels\annotation.csv
Reference-label source: ai_assisted
Working reference counts: {'negative': 104, 'neutral': 110, 'positive': 17}
Labels are complete; analysis and evaluation can run without group annotation.


## 5. Fit three LDA topics and compute rule sentiment

The analyzer uses three LDA topics with a fixed seed. LDA uses word counts; proximity uses normalized TF-IDF and token sets. The fixed English/Filipino/Taglish lexical baseline predicts positive, negative, or neutral sentiment and stores its evidence.

Scores are heuristic, not calibrated confidence. Rule matches can reflect general emotional tone rather than target-related polarity; zero-cue neutral predictions are fallback classifications, not confirmed neutral opinion. Evaluation against the supplied independent AI-assisted references measures these discrepancies.

Institutional announcements, news reporting, and personal opinions have different meanings for interpretation. Inspect content purpose in the annotation decisions and source records before describing results as public approval. Per-record predictions are displayed because supplied annotation is already complete.

Compare the **supplied reference** sentiment summary below with the **rule-prediction** summary. The reference summary groups by `content_kind` (assisted assessment of news/commentary purpose), topic, platform, and label. Its references can use available public embed context; the rule model uses the canonical post text only. That input difference can explain some evaluation errors. If the working labels are replaced with a separately reviewed corpus, the supplied reference summary remains an audit of the original snapshot and should not be reported as the revised run.

In [6]:
SHOW_DOCUMENT_PREDICTIONS = True
analysis_outputs = tuple(
    ANALYSIS_DIR / filename
    for filename in (
        "analyzed_documents.csv",
        "topic_terms.csv",
        "topic_metrics.csv",
        "sentiment_summary.csv",
        "proximity_pairs.csv",
    )
)
analysis_fingerprint = stage_fingerprint("analysis", (CONFIG, PROCESSED))
if not cached_stage("analysis", analysis_outputs, analysis_fingerprint):
    run_cli(
        "analyze",
        "--config",
        str(CONFIG),
        "--input",
        str(PROCESSED),
        "--output-dir",
        str(ANALYSIS_DIR),
        *(["--overwrite"] if FORCE_REBUILD else []),
    )
    remember_stage("analysis", analysis_outputs, analysis_fingerprint)
for filename in ("topic_terms.csv", "topic_metrics.csv", "sentiment_summary.csv"):
    preview_csv(ANALYSIS_DIR / filename)
if SHOW_DOCUMENT_PREDICTIONS:
    preview_csv(ANALYSIS_DIR / "analyzed_documents.csv", max_columns=12)

print("Supplied AI-assisted reference polarity summary (separate from rule predictions):")
preview_csv(SNAPSHOT / "labels" / "reference_sentiment_summary.csv", limit=12, max_columns=10)

documents -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\analysis\analyzed_documents.csv
topic_terms -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\analysis\topic_terms.csv
topic_metrics -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\analysis\topic_metrics.csv
sentiment_summary -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\analysis\sentiment_summary.csv
proximity_pairs -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\analysis\proximity_pairs.csv
data\labeled-notebook-run\analysis\topic_terms.csv: 30 rows


topic_id,rank,term,probability
0,1,philippine,0.017016792449530922
0,2,government,0.014802508883610187
0,3,philippines,0.01221341434940561
0,4,marcos,0.01030403576597827
0,5,just,0.005972543039551751
0,6,discord,0.0059255479242579494


data\labeled-notebook-run\analysis\topic_metrics.csv: 9 rows


metric,topic_id,value,evaluation_scope,notes
npmi_coherence,0,-0.09912721253302899,in_sample_document_presence,"Unseen pairs=-1; universal pairs=0; all top-term pairs are averaged. Topic/proximity features omit stopwords, emoji-only tokens, and the exact repeated IA/Users clickbait publisher..."
npmi_coherence,1,0.19789684051554016,in_sample_document_presence,"Unseen pairs=-1; universal pairs=0; all top-term pairs are averaged. Topic/proximity features omit stopwords, emoji-only tokens, and the exact repeated IA/Users clickbait publisher..."
npmi_coherence,2,0.16196060509656715,in_sample_document_presence,"Unseen pairs=-1; universal pairs=0; all top-term pairs are averaged. Topic/proximity features omit stopwords, emoji-only tokens, and the exact repeated IA/Users clickbait publisher..."
lda_perplexity,,2057.6061953500734,in_sample,"In-sample fit diagnostic; not held-out topic quality. Topic/proximity features omit stopwords, emoji-only tokens, and the exact repeated IA/Users clickbait publisher template; docu..."
mean_npmi_coherence,,0.08691007769302611,in_sample,"Mean of topic NPMI values. Topic/proximity features omit stopwords, emoji-only tokens, and the exact repeated IA/Users clickbait publisher template; document and sentiment text are..."
topic_diversity,,0.6333333333333333,in_sample,"Unique top terms / all selected top-term slots. Topic/proximity features omit stopwords, emoji-only tokens, and the exact repeated IA/Users clickbait publisher template; document a..."


data\labeled-notebook-run\analysis\sentiment_summary.csv: 84 rows


topic_id,platform,record_type,month,label,count,total,share
0,bluesky,post,2026-08,negative,0,10,0.0
0,bluesky,post,2026-08,neutral,10,10,1.0
0,bluesky,post,2026-08,positive,0,10,0.0
0,bluesky,post,2026-09,negative,3,34,0.08823529411764706
0,bluesky,post,2026-09,neutral,26,34,0.7647058823529411
0,bluesky,post,2026-09,positive,5,34,0.14705882352941177


data\labeled-notebook-run\analysis\analyzed_documents.csv: 231 rows


record_id,platform,created_at,text,source_url,query,record_type,collected_at,gold_label,clean_text,topic_id,topic_probability
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,2026-10-12T07:02:57.396000+00:00,"the us and japan are violating the philippines' sovereignty in recent years, the marcos administration has raised the banner of ""armed forces modernization",https://bsky.app/profile/did:plc:bmnkh5p5evoqb7iwrx362p4t/post/3mxnx6uq7d22a,Philippines Marcos,post,2026-10-12T07:37:19.329647Z,neutral,"the us and japan are violating the philippines' sovereignty in recent years, the marcos administration has raised the banner of ""armed forces modernization",0,0.9480004354380006
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx62vedk2a,bluesky,2026-10-12T07:02:30.302000+00:00,"political battles are tearing the philippines apart the illusion of arms procurement: how the marcos administration drains the national treasury in the name of ""military modernizat...",https://bsky.app/profile/did:plc:bmnkh5p5evoqb7iwrx362p4t/post/3mxnx62vedk2a,Philippines Marcos,post,2026-10-12T07:37:19.329647Z,negative,"political battles are tearing the philippines apart the illusion of arms procurement: how the marcos administration drains the national treasury in the name of ""military modernizat...",0,0.9551096410854494
bluesky:at://did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/app.bsky.feed.post/3mxnufw5l522l,bluesky,2026-10-12T06:13:12.126989+00:00,president marcos returns from singapore visit focused on business expansion and energy investments 🤖 ia: it's not clickbait ✅ 👥 users: it's not clickbait ✅ marcos singapore investm...,https://bsky.app/profile/did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/post/3mxnufw5l522l,Philippines Marcos,post,2026-10-12T07:37:19.329647Z,neutral,president marcos returns from singapore visit focused on business expansion and energy investments 🤖 ia: it's not clickbait ✅ 👥 users: it's not clickbait ✅ marcos singapore investm...,1,0.9498494050420829
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxnqzmxal322,bluesky,2026-10-12T05:12:39+00:00,marcos orders scrutiny of alleged p319.33 million china remittances linked to duterte's husband philippines politics,https://bsky.app/profile/did:plc:jhgdghatjzlq2nzwawreyfll/post/3mxnqzmxal322,Philippines Marcos,post,2026-10-12T07:37:19.329647Z,negative,marcos orders scrutiny of alleged p319.33 million china remittances linked to duterte's husband philippines politics,0,0.948819138377135
bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxnek4papo2w,bluesky,2026-10-12T01:29:12.807000+00:00,"japanese ambassador katsuro kitagawa presented his letters of credence to president ferdinand marcos jr. on oct. 1, formally beginning his tenure as japan's ambassador to the phili...",https://bsky.app/profile/did:plc:qvidzz7dm76lanyikijwp4c4/post/3mxnek4papo2w,Philippines Marcos,post,2026-10-12T07:37:19.329647Z,neutral,"japanese ambassador katsuro kitagawa presented his letters of credence to president ferdinand marcos jr. on oct. 1, formally beginning his tenure as japan's ambassador to the phili...",1,0.9687668313265183
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxn2o72ay222,bluesky,2026-10-11T22:32:33+00:00,marcos rejects chinese permission demands after nearly 10 hours to evacuate ayungin sailor china philippines politics,https://bsky.app/profile/did:plc:jhgdghatjzlq2nzwawreyfll/post/3mxn2o72ay222,Philippines Marcos,post,2026-10-12T07:37:19.329647Z,neutral,marcos rejects chinese permission demands after nearly 10 hours to evacuate ayungin sailor china philippines politics,1,0.9512282181090235


Supplied AI-assisted reference polarity summary (separate from rule predictions):
deliverables\labeled_2026\labels\reference_sentiment_summary.csv: 34 rows


topic_id,platform,content_kind,label,count,total,share,label_source,notes
0,bluesky,news_report,negative,6,37,0.16216216216216217,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,bluesky,news_report,neutral,28,37,0.7567567567567568,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,bluesky,news_report,positive,3,37,0.08108108108108109,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,bluesky,other,negative,2,2,1.0,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,bluesky,personal_commentary,negative,28,33,0.8484848484848485,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,bluesky,personal_commentary,neutral,4,33,0.12121212121212122,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,bluesky,personal_commentary,positive,1,33,0.030303030303030304,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,mastodon,news_report,negative,3,6,0.5,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,mastodon,news_report,neutral,1,6,0.16666666666666666,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."
0,mastodon,news_report,positive,2,6,0.3333333333333333,ai_assisted,"Supplied target-polarity reference labels, not rule predictions or population approval."


### Topic interpretation and topic-model evaluation

Name each topic only after reading its top terms and representative posts. Check overlapping terms, poorly assigned documents, and whether three topics are actually interpretable.

`topic_metrics.csv` records document-presence NPMI coherence, top-term diversity, and explicitly **in-sample** perplexity. These are descriptive diagnostics; they do not establish topic validity or predictive generalization.

| Topic | Name supported by records | Representative record IDs | Interpretation and limitations |
| --- | --- | --- | --- |
| 0 | Complete after inspection | Complete after inspection | Complete after inspection |
| 1 | Complete after inspection | Complete after inspection | Complete after inspection |
| 2 | Complete after inspection | Complete after inspection | Complete after inspection |

Report values from the generated topic metrics, together with the human review. Never replace missing topic output with assumed results.

## 6. Examine cosine, Jaccard, and Euclidean proximity

Larger cosine and Jaccard similarity mean more similar wording under their respective representations. Smaller Euclidean distance means closer normalized TF-IDF vectors. Similar wording does not establish agreement, accuracy, shared authorship, or coordinated behavior.

The pair limit bounds computation, so the CSV may contain only selected pairs. Inspect `pair_status` and exclude empty-representation results from substantive interpretation.

In [7]:
preview_csv(ANALYSIS_DIR / "proximity_pairs.csv", limit=10)

data\labeled-notebook-run\analysis\proximity_pairs.csv: 5,000 rows


record_id_a,platform_a,record_id_b,platform_b,cosine_similarity,jaccard_similarity,euclidean_distance,pair_status
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxitkvrikb2t,bluesky,0.02218533029622232,0.07142857142857142,1.3984381786148272,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxg4tyooco2k,bluesky,0.01774295790022464,0.06451612903225806,1.4016112457452496,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:6oue7bcx5zgbpkl5upsuadwt/app.bsky.feed.post/3mxegreefgr23,bluesky,0.013298366137425334,0.04878048780487805,1.4047787255383495,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:waoxtoj7342xkpen7phiidus/app.bsky.feed.post/3mx5rixcrgc2y,bluesky,0.021778786774340052,0.09523809523809523,1.3987288609488688,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:ni4mtuacxk2ksyfftzm6vxmj/app.bsky.feed.post/3mx5gbaqed22o,bluesky,0.014591061919563577,0.05,1.4038582108464062,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:dxe2wcmkv5ffnmu4qutgmcgq/app.bsky.feed.post/3mx3vupwmzs2p,bluesky,0.10250103851947492,0.14285714285714285,1.3397753255531504,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:5oy2xcpdflrexpgdw5wysnjd/app.bsky.feed.post/3mx3f2nfftc2m,bluesky,0.03557232379376696,0.1,1.3888323701629604,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:mifsalmyx5di7yuxkgkjbh5j/app.bsky.feed.post/3mwwep2ku7c2o,bluesky,0.016679053653106034,0.06060606060606061,1.402370098331317,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:2qjij4etp5jrqyelyynq7eoc/app.bsky.feed.post/3mxnm54omwv26,bluesky,0.0,0.0,1.4142135623730951,ok
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxmvfvnipl22,bluesky,0.02076630751336665,0.04,1.3994525304465553,ok


### Proximity interpretation

Select actual pairs from the CSV and inspect both underlying posts:

- Pair IDs and the three measured values:
- Shared and differing wording that explains the comparison:
- Whether text similarity corresponds to stance similarity in this example:
- Pair-status and sampling limitations:

Keep examples traceable to record IDs. Do not quote unnecessary personal information in the submission.

## 7. Evaluate: stratified 90/10 holdout and 10-fold development CV

Evaluation runs by default against the supplied **AI-assisted reference labels**. Report the resulting metrics as reference-label agreement, not human-validated accuracy. The evaluator reserves a stratified 10% holdout first, then uses ten stratified folds within the 90% development portion. It validates all three classes and sufficient examples per development class.

The fixed rule classifier has no training step. Cross-validation measures score variation across development partitions; it does not show learning on training folds. Keep holdout metrics separate. After using holdout errors to revise the model, obtain a fresh untouched holdout for a final assessment.

The annotation file is reloaded on every evaluation. Stale or incomplete outputs fail the hash checks. Optional human-reviewed replacement labels require a documented review and `LABEL_SOURCE = "human"`; they are preserved rather than overwritten.

In [8]:
RUN_EVALUATION = True
evaluation_outputs = tuple(
    EVALUATION_DIR / filename
    for filename in (
        "evaluation_metrics.csv",
        "classification_report.csv",
        "confusion_matrix.csv",
        "split_assignments.csv",
        "evaluation_predictions.csv",
    )
)
annotation_records = verify_annotation_records(
    ANNOTATIONS, prepared_records, require_supplied=LABEL_SOURCE == "ai_assisted"
)
if not RUN_EVALUATION:
    print("Evaluation disabled explicitly for this run.")
else:
    evaluation_fingerprint = stage_fingerprint("evaluation", (CONFIG, ANNOTATIONS), LABEL_SOURCE)
    if not cached_stage("evaluation", evaluation_outputs, evaluation_fingerprint):
        run_cli(
            "evaluate",
            "--config",
            str(CONFIG),
            "--input",
            str(ANNOTATIONS),
            "--output-dir",
            str(EVALUATION_DIR),
            "--label-source",
            LABEL_SOURCE,
            *(["--overwrite"] if FORCE_REBUILD else []),
        )
        remember_stage("evaluation", evaluation_outputs, evaluation_fingerprint)
    print(
        "Evaluation scope: AI-assisted reference agreement, not human-validated accuracy."
        if LABEL_SOURCE == "ai_assisted"
        else "Evaluation scope: documented human reference labels."
    )
    for scope, title in (
        ("holdout", "Held-out reference agreement"),
        ("cv_mean", "Development cross-validation mean"),
        ("cv_std", "Development cross-validation standard deviation"),
    ):
        print(title)
        preview_csv(EVALUATION_DIR / "evaluation_metrics.csv", limit=5, where={"split": scope})
    print("Held-out per-class reference agreement")
    preview_csv(
        EVALUATION_DIR / "classification_report.csv",
        limit=3,
        max_columns=9,
        where={"split": "holdout"},
    )
    print("Held-out confusion matrix")
    preview_csv(EVALUATION_DIR / "confusion_matrix.csv", limit=9, where={"split": "holdout"})
    for filename in ("split_assignments.csv", "evaluation_predictions.csv"):
        preview_csv(EVALUATION_DIR / filename)

evaluation_metrics -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\evaluation\evaluation_metrics.csv
classification_report -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\evaluation\classification_report.csv
confusion_matrix -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\evaluation\confusion_matrix.csv
split_assignments -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\evaluation\split_assignments.csv
evaluation_predictions -> C:\Users\paulp\OneDrive\Documents\VSCODE_REPO\social-media-scraper\tmp\labeled-notebook-qa-final\checkout\data\labeled-notebook-run\evaluation\evaluation_predictions.csv
Evaluation scope: AI-assisted reference agreement, n

split,fold,n_records,metric,value,model_version,label_source,notes
holdout,,24,accuracy,0.4583333333333333,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,24,macro_precision,0.16666666666666666,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,24,macro_recall,0.3333333333333333,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,24,macro_f1,0.2222222222222222,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,24,balanced_accuracy,0.3333333333333333,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...


Development cross-validation mean
data\labeled-notebook-run\evaluation\evaluation_metrics.csv: 5 matching rows of 70


split,fold,n_records,metric,value,model_version,label_source,notes
cv_mean,,207,accuracy,0.5126190476190475,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
cv_mean,,207,macro_precision,0.464229302832244,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
cv_mean,,207,macro_recall,0.3733333333333334,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
cv_mean,,207,macro_f1,0.31165795889933817,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
cv_mean,,207,balanced_accuracy,0.3733333333333334,en-fil-taglish-2026.1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...


Development cross-validation standard deviation
data\labeled-notebook-run\evaluation\evaluation_metrics.csv: 5 matching rows of 70


...

split,fold,actual_label,predicted_label,count,label_source,notes
holdout,,negative,negative,0,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,negative,neutral,10,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,negative,positive,1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,neutral,negative,0,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,neutral,neutral,11,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,neutral,positive,0,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,positive,negative,1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,positive,neutral,1,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...
holdout,,positive,positive,0,ai_assisted,Agreement against AI-assisted reference annotations; not human-validated accuracy. Fixed lexicon model: no training or tuning; CV measures variation across development partitions. ...


data\labeled-notebook-run\evaluation\split_assignments.csv: 231 rows


record_id,platform,split,validation_fold,seed,holdout_fraction,cv_folds,label_source
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,development,4,42,0.1,10,ai_assisted
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx62vedk2a,bluesky,development,6,42,0.1,10,ai_assisted
bluesky:at://did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/app.bsky.feed.post/3mxnufw5l522l,bluesky,development,2,42,0.1,10,ai_assisted
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxnqzmxal322,bluesky,development,1,42,0.1,10,ai_assisted
bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxnek4papo2w,bluesky,development,8,42,0.1,10,ai_assisted
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxn2o72ay222,bluesky,development,4,42,0.1,10,ai_assisted


data\labeled-notebook-run\evaluation\evaluation_predictions.csv: 231 rows


record_id,platform,split,fold,gold_label,predicted_label,sentiment_score,sentiment_evidence
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx6uq7d22a,bluesky,development,4,neutral,neutral,0.0,[]
bluesky:at://did:plc:bmnkh5p5evoqb7iwrx362p4t/app.bsky.feed.post/3mxnx62vedk2a,bluesky,development,6,negative,neutral,0.0,[]
bluesky:at://did:plc:yhjtgrqaoz2gu2qhg4eaqeqk/app.bsky.feed.post/3mxnufw5l522l,bluesky,development,2,neutral,neutral,0.0,[]
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxnqzmxal322,bluesky,development,1,negative,neutral,0.0,[]
bluesky:at://did:plc:qvidzz7dm76lanyikijwp4c4/app.bsky.feed.post/3mxnek4papo2w,bluesky,development,8,neutral,neutral,0.0,[]
bluesky:at://did:plc:jhgdghatjzlq2nzwawreyfll/app.bsky.feed.post/3mxn2o72ay222,bluesky,development,4,neutral,neutral,0.0,[]


### Evaluation interpretation

Complete these notes from the actual CSVs:

- Number and class balance of supplied references; blinded disjoint annotation, independent random-sample AI review, actual agreement, and adjudication:
- Development-fold mean/variation and holdout accuracy, macro precision, recall, F1, and balanced accuracy:
- Per-class confusion and representative development errors:
- Model version, split reproducibility, reporting-versus-personal-opinion distinctions, and reference-label limitations:

Use CSV scope and label-source fields to distinguish development/holdout and AI-assisted/human-reviewed evaluation. Metric names such as `accuracy` describe agreement with these reference labels; they do not establish human-validated correctness or actual government performance.

The tables above display actual held-out metrics and class errors separately from development CV mean/variation. Consult `deliverables/labeled_2026/results_notes.md` for supplied model/topic diagnostics; use the real generated values and retain the reference-label and sampling limits in the report.

## 8. Scripts, outputs, and remarks

All generated tabular datasets are CSV. The supplied curated and labeled snapshot remains unchanged; working outputs go to `data/labeled-notebook-run/`.

| Script / stage | Output | Remarks |
| --- | --- | --- |
| Supplied public API collection | `deliverables/labeled_2026/raw/*.csv` | Actual returned records and collection provenance. |
| Supplied assisted curation/context review | `processed/posts.csv`, annotation decisions/context/review/adjudication CSVs | Curated input, reasons, available context, and actual AI consistency diagnostics. |
| Supplied reference summary | `annotation_summary.csv`, `labels/reference_sentiment_summary.csv` | Reference polarity by topic/source/content purpose; separate from rule predictions. |
| Supplied reference-label copy | `data/labeled-notebook-run/labels/annotation.csv` | Complete independent AI-assisted reference labels; working edits are preserved. |
| `ph_sentiment analyze` | `analyzed_documents.csv`, `topic_terms.csv`, `topic_metrics.csv` | Three requested LDA topics, fixed-rule predictions, and topic diagnostics. |
| Same analysis command | `proximity_pairs.csv`, `sentiment_summary.csv` | Cosine, Jaccard, Euclidean proximity and descriptive counts/shares. |
| `ph_sentiment evaluate --label-source ai_assisted` | Metrics, reports, confusion matrices, split assignments, predictions | Default 90/10 split and ten development folds; agreement with AI-assisted references. |
| This notebook | Executed cells, tables, and interpretation | Complete discussion and rename a submission copy to `GROUP NAME.ipynb`. |

The group can focus on documentation, topic names/representative examples, proximity interpretation, evaluation limitations, the final discussion, and peer evaluation. No manual labeling is required for the supplied computational handoff. Keep all reported outputs traceable to this snapshot and describe the AI-assisted references honestly.

Platform users self-select; APIs restrict coverage; repeated content, source imbalance, language, sarcasm, and target errors can distort results. Sentiment reflects this sampled discussion. Specific claims about actual government performance need separate evidence.